# Level 3 — Task 2: Risk Analysis & Fraud Detection**Internship:** Codveda Technology — Business Analytics  **Objective:** Identify risks and detect fraudulent/anomalous activities using ML.| Section | Technique | Goal ||---------|-----------|------|| Anomaly Detection | Isolation Forest + Z-Score + IQR | Flag abnormal usage patterns || Risk Scoring | Weighted scorecard | Rank customers by risk tier || Fraud Detection ML | Random Forest + GBM + LR | Classify high-risk behaviours || Predictive Risk Mgmt | Threshold optimisation | Proactive intervention strategy |---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os
warnings.filterwarnings('ignore')

from sklearn.ensemble        import IsolationForest, RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model    import LogisticRegression
from sklearn.preprocessing   import StandardScaler, LabelEncoder
from sklearn.impute          import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.decomposition   import PCA
from sklearn.metrics         import (classification_report, confusion_matrix,
                                     roc_auc_score, roc_curve, ConfusionMatrixDisplay,
                                     precision_recall_curve, average_precision_score)
from scipy import stats

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
np.random.seed(42)
print('Libraries loaded')

In [ ]:
DATA_DIR = '../data'
churn = pd.read_csv(os.path.join(DATA_DIR, 'churn_cleaned.csv'))
churn['Churn_label'] = churn['Churn'].map({1:'Churned',0:'Retained',True:'Churned',False:'Retained'})
churn['Churn_int']   = (churn['Churn_label']=='Churned').astype(int)
print(f'Churn dataset: {churn.shape}')
display(churn.head(3))

---
## 1. Anomaly Detection
Three methods: Z-Score, IQR, and Isolation Forest (multivariate ML).


In [ ]:
from scipy.stats import zscore

def detect_anomalies_zscore(df, col, threshold=3.0):
    z = np.abs(zscore(df[col].dropna()))
    n = (z > threshold).sum()
    print(f'[Z-Score] {col}: {n} anomalies ({n/len(df)*100:.2f}%) at |z|>{threshold}')
    return z > threshold

def detect_anomalies_iqr(df, col, factor=1.5):
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    lo, hi = Q1 - factor*IQR, Q3 + factor*IQR
    mask = (df[col] < lo) | (df[col] > hi)
    print(f'[IQR]     {col}: {mask.sum()} anomalies ({mask.mean()*100:.2f}%)')
    return mask

print('=== Total Day Charge ===')
detect_anomalies_zscore(churn, 'Total day charge')
detect_anomalies_iqr(churn, 'Total day charge')
print('=== Customer Service Calls ===')
detect_anomalies_zscore(churn, 'Customer service calls')
detect_anomalies_iqr(churn, 'Customer service calls')

In [ ]:
anom_features = ['Total day minutes','Total day charge','Total intl minutes',
                  'Total intl charge','Customer service calls','Total eve charge']
X_anom = pd.DataFrame(
    SimpleImputer(strategy='median').fit_transform(churn[anom_features]),
    columns=anom_features)
sc_a   = StandardScaler()
X_a_sc = sc_a.fit_transform(X_anom)
iso    = IsolationForest(n_estimators=200, contamination=0.05, random_state=42)
anom_lbl   = iso.fit_predict(X_a_sc)
anom_score = iso.decision_function(X_a_sc)
X_anom['is_anomaly'] = (anom_lbl == -1).astype(int)
n_anom = (anom_lbl == -1).sum()
print(f'Isolation Forest: {n_anom} anomalies ({n_anom/len(X_anom)*100:.1f}%)')

In [ ]:
pca_a   = PCA(n_components=2, random_state=42)
X_pca_a = pca_a.fit_transform(X_a_sc)
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
nm, am = (anom_lbl == 1), (anom_lbl == -1)
axes[0].scatter(X_pca_a[nm,0], X_pca_a[nm,1], c='#3498db', alpha=0.3, s=12, label='Normal')
axes[0].scatter(X_pca_a[am,0], X_pca_a[am,1], c='#e74c3c', alpha=0.8, s=35, marker='x', label='Anomaly')
axes[0].set_title('Isolation Forest - PCA View', fontweight='bold'); axes[0].legend()
axes[1].hist(anom_score[nm], bins=40, color='#3498db', alpha=0.7, edgecolor='white', label='Normal', density=True)
axes[1].hist(anom_score[am], bins=20, color='#e74c3c', alpha=0.8, edgecolor='white', label='Anomaly', density=True)
axes[1].axvline(x=0, color='black', linestyle='--')
axes[1].set_title('Anomaly Score Distribution', fontweight='bold'); axes[1].legend()
axes[2].scatter(X_anom.loc[X_anom['is_anomaly']==0,'Total day charge'],
                X_anom.loc[X_anom['is_anomaly']==0,'Customer service calls'],
                c='#3498db', alpha=0.3, s=12)
axes[2].scatter(X_anom.loc[X_anom['is_anomaly']==1,'Total day charge'],
                X_anom.loc[X_anom['is_anomaly']==1,'Customer service calls'],
                c='#e74c3c', alpha=0.8, s=40, marker='x')
axes[2].set_xlabel('Total Day Charge'); axes[2].set_ylabel('Service Calls')
axes[2].set_title('Anomalies: Charge vs Calls', fontweight='bold')
plt.suptitle('Anomaly Detection - Isolation Forest', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('anomaly_detection_ml.png', bbox_inches='tight')
plt.show()

---
## 2. Customer Risk Scorecard
Weighted rules-based scoring (0-100). Higher score = higher churn risk.


In [ ]:
def build_risk_scorecard(df):
    d = df.copy(); d['risk_score'] = 0.0
    d.loc[d['International plan'].isin([1,'Yes',True]), 'risk_score'] += 25
    d.loc[d['Customer service calls'] >= 4, 'risk_score'] += 25
    d.loc[d['Total day minutes'] > d['Total day minutes'].quantile(0.75), 'risk_score'] += 15
    d.loc[d['Voice mail plan'].isin([0,'No',False]), 'risk_score'] += 10
    d.loc[d['Account length'] < d['Account length'].quantile(0.25), 'risk_score'] += 15
    d.loc[d['Total intl minutes'] > d['Total intl minutes'].quantile(0.75), 'risk_score'] += 10
    d['risk_tier'] = pd.cut(d['risk_score'], bins=[-1,20,40,60,100],
                            labels=['Low Risk','Medium Risk','High Risk','Very High Risk'])
    return d

churn_risk = build_risk_scorecard(churn)
tier_analysis = churn_risk.groupby('risk_tier', observed=True)['Churn_int'].agg(
    customers='count', churned='sum',
    churn_rate=lambda x: f'{x.mean()*100:.1f}%').reset_index()
print('Churn Rate by Risk Tier:')
display(tier_analysis)

In [ ]:
tier_colors = {'Low Risk':'#2ecc71','Medium Risk':'#f39c12','High Risk':'#e67e22','Very High Risk':'#e74c3c'}
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
axes[0].hist(churn_risk[churn_risk['Churn_int']==0]['risk_score'], bins=20,
             color='#3498db', alpha=0.7, edgecolor='white', density=True, label='Retained')
axes[0].hist(churn_risk[churn_risk['Churn_int']==1]['risk_score'], bins=20,
             color='#e74c3c', alpha=0.7, edgecolor='white', density=True, label='Churned')
axes[0].set_title('Risk Score Distribution', fontweight='bold'); axes[0].legend()
tc = churn_risk['risk_tier'].value_counts().sort_index()
axes[1].bar(tc.index, tc.values, color=[tier_colors[t] for t in tc.index], edgecolor='white')
for i, v in enumerate(tc.values): axes[1].text(i, v+20, str(v), ha='center', fontweight='bold', fontsize=9)
axes[1].set_title('Customers per Tier', fontweight='bold'); axes[1].tick_params(axis='x', rotation=15)
tic = churn_risk.groupby('risk_tier', observed=True)['Churn_int'].mean() * 100
bars = axes[2].bar(tic.index, tic.values, color=[tier_colors[t] for t in tic.index], edgecolor='white')
for bar, val in zip(bars, tic.values):
    axes[2].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5, f'{val:.1f}%', ha='center', fontweight='bold')
axes[2].set_title('Churn Rate by Tier (Validation)', fontweight='bold'); axes[2].tick_params(axis='x', rotation=15)
plt.suptitle('Customer Risk Scorecard', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('risk_scorecard.png', bbox_inches='tight')
plt.show()

---
## 3. ML-Based Fraud Detection
Evaluated on Precision-Recall curves (better metric for imbalanced fraud data).


In [ ]:
fd = churn.copy()
drop_cols = [c for c in ['State','Area code','split','Churn_label'] if c in fd.columns]
fd = fd.drop(columns=drop_cols)
for col in fd.select_dtypes(include=['object','bool']).columns:
    if col not in ['Churn_int']:
        fd[col] = LabelEncoder().fit_transform(fd[col].astype(str))
fd['total_charge']    = fd['Total day charge'] + fd['Total eve charge'] + fd['Total night charge'] + fd['Total intl charge']
fd['charge_per_call'] = fd['total_charge'] / (fd['Total day calls'] + fd['Total eve calls'] + fd['Total night calls'] + 1)
fd['intl_ratio']      = fd['Total intl minutes'] / (fd['Total day minutes'] + 1)
X_fd = fd.drop(columns=['Churn','Churn_int'], errors='ignore')
y_fd = fd['Churn_int'] if 'Churn_int' in fd.columns else fd['Churn'].astype(int)
X_fd = pd.DataFrame(SimpleImputer(strategy='median').fit_transform(X_fd), columns=X_fd.columns)
X_tr, X_te, y_tr, y_te = train_test_split(X_fd, y_fd, test_size=0.2, stratify=y_fd, random_state=42)
sc_fd = StandardScaler()
X_tr_sc = sc_fd.fit_transform(X_tr); X_te_sc = sc_fd.transform(X_te)
print(f'Fraud rate: {y_fd.mean()*100:.1f}% | Train: {X_tr.shape} | Test: {X_te.shape}')

In [ ]:
fraud_models = {
    'Random Forest'      : RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42),
    'Gradient Boosting'  : GradientBoostingClassifier(n_estimators=100, random_state=42),
    'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
}
fraud_results, fraud_probs, fraud_fitted = [], {}, {}
for name, model in fraud_models.items():
    model.fit(X_tr_sc, y_tr)
    y_pred = model.predict(X_te_sc)
    y_prob = model.predict_proba(X_te_sc)[:,1]
    rep = classification_report(y_te, y_pred, output_dict=True)
    ap  = average_precision_score(y_te, y_prob)
    fraud_results.append({'Model':name,'Accuracy':round(rep['accuracy'],4),
                          'ROC_AUC':round(roc_auc_score(y_te,y_prob),4),'Avg_Prec':round(ap,4)})
    fraud_probs[name] = y_prob; fraud_fitted[name] = model
fr_df = pd.DataFrame(fraud_results).set_index('Model')
print('Fraud Detection Comparison:')
display(fr_df.sort_values('ROC_AUC', ascending=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
colors = ['#e74c3c','#3498db','#2ecc71']
for (name, prob), color in zip(fraud_probs.items(), colors):
    fpr, tpr, _ = roc_curve(y_te, prob); auc = roc_auc_score(y_te, prob)
    axes[0].plot(fpr, tpr, color=color, linewidth=2.5, label=f'{name} AUC={auc:.3f}')
    prec, rec, _ = precision_recall_curve(y_te, prob); ap = average_precision_score(y_te, prob)
    axes[1].plot(rec, prec, color=color, linewidth=2.5, label=f'{name} AP={ap:.3f}')
axes[0].plot([0,1],[0,1],'k--',linewidth=1.2)
axes[0].set_title('ROC Curves', fontweight='bold'); axes[0].legend(fontsize=9)
axes[1].axhline(y=y_te.mean(), color='gray', linestyle='--')
axes[1].set_title('Precision-Recall Curves', fontweight='bold'); axes[1].legend(fontsize=9)
plt.suptitle('Fraud Detection Model Evaluation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fraud_detection_models.png', bbox_inches='tight')
plt.show()

---
## 4. Threshold Optimisation & Risk Strategy


In [ ]:
def optimise_threshold(y_true, y_prob, fp_cost=5, fn_cost=50):
    thresholds = np.arange(0.01, 1.0, 0.01)
    f1s, costs = [], []
    for t in thresholds:
        preds = (y_prob >= t).astype(int)
        rep   = classification_report(y_true, preds, output_dict=True, zero_division=0)
        f1s.append(rep['1']['f1-score'])
        cm    = confusion_matrix(y_true, preds)
        costs.append(cm[0,1]*fp_cost + cm[1,0]*fn_cost if cm.shape==(2,2) else np.nan)
    return thresholds[np.argmax(f1s)], thresholds[np.nanargmin(costs)], thresholds, f1s, costs

best_probs = fraud_probs['Random Forest']
opt_t, min_cost_t, thresholds, f1s, costs = optimise_threshold(y_te, best_probs)
print(f'Optimal F1 threshold: {opt_t:.2f} | Min-cost threshold: {min_cost_t:.2f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(thresholds, f1s, '#e74c3c', linewidth=2.5, label='F1 Score')
axes[0].axvline(x=opt_t, color='black', linestyle='--', linewidth=2, label=f'Optimal={opt_t:.2f}')
axes[0].axvline(x=0.5, color='gray', linestyle=':', linewidth=1.5, label='Default=0.5')
axes[0].set_title('F1 vs Threshold', fontweight='bold'); axes[0].legend()
axes[1].plot(thresholds, costs, '#9b59b6', linewidth=2.5)
axes[1].axvline(x=min_cost_t, color='#e74c3c', linestyle='--', linewidth=2, label=f'Min cost={min_cost_t:.2f}')
axes[1].set_title('Business Cost vs Threshold (FP=$5, FN=$50)', fontweight='bold'); axes[1].legend()
plt.suptitle('Threshold Optimisation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('threshold_optimisation.png', bbox_inches='tight')
plt.show()

In [ ]:
all_probs = fraud_fitted['Random Forest'].predict_proba(sc_fd.transform(X_fd))[:,1]
churn_risk['fraud_risk_score'] = all_probs[:len(churn_risk)]
churn_risk['fraud_flag'] = (all_probs[:len(churn_risk)] >= min_cost_t).astype(int)
churn_risk['total_charge'] = (churn['Total day charge'] + churn['Total eve charge'] +
                               churn['Total night charge'] + churn['Total intl charge'])
flagged = churn_risk[churn_risk['fraud_flag']==1]
print(f'Flagged: {len(flagged):,} customers | Precision: {flagged["Churn_int"].mean()*100:.1f}% | Revenue at risk: ${flagged["total_charge"].sum():,.0f}')

band_colors = ['#e74c3c','#e67e22','#f39c12','#2ecc71']
bands = {'Critical\n(>0.8)':(churn_risk['fraud_risk_score']>0.8).sum(),
         'High\n(0.6-0.8)':((churn_risk['fraud_risk_score']>0.6)&(churn_risk['fraud_risk_score']<=0.8)).sum(),
         'Medium\n(0.4-0.6)':((churn_risk['fraud_risk_score']>0.4)&(churn_risk['fraud_risk_score']<=0.6)).sum(),
         'Low\n(<0.4)':(churn_risk['fraud_risk_score']<=0.4).sum()}
rev_b = [churn_risk[churn_risk['fraud_risk_score']>0.8]['total_charge'].sum(),
         churn_risk[(churn_risk['fraud_risk_score']>0.6)&(churn_risk['fraud_risk_score']<=0.8)]['total_charge'].sum(),
         churn_risk[(churn_risk['fraud_risk_score']>0.4)&(churn_risk['fraud_risk_score']<=0.6)]['total_charge'].sum(),
         churn_risk[churn_risk['fraud_risk_score']<=0.4]['total_charge'].sum()]

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
axes[0].hist(churn_risk[churn_risk['Churn_int']==0]['fraud_risk_score'], bins=40,
             color='#3498db', alpha=0.7, edgecolor='white', density=True, label='Retained')
axes[0].hist(churn_risk[churn_risk['Churn_int']==1]['fraud_risk_score'], bins=40,
             color='#e74c3c', alpha=0.7, edgecolor='white', density=True, label='Churned')
axes[0].axvline(min_cost_t, color='black', linestyle='--', linewidth=2)
axes[0].set_title('ML Risk Score Distribution', fontweight='bold'); axes[0].legend()
axes[1].bar(bands.keys(), bands.values(), color=band_colors, edgecolor='white')
for i, v in enumerate(bands.values()): axes[1].text(i, v+10, str(v), ha='center', fontweight='bold', fontsize=9)
axes[1].set_title('Intervention Priority Bands', fontweight='bold')
axes[2].bar(['Critical','High','Medium','Low'], rev_b, color=band_colors, edgecolor='white')
for i, v in enumerate(rev_b): axes[2].text(i, v+300, f'${v:,.0f}', ha='center', fontweight='bold', fontsize=8)
axes[2].set_title('Revenue at Risk by Band', fontweight='bold')
plt.suptitle('Predictive Risk Management Strategy', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('risk_management_strategy.png', bbox_inches='tight')
plt.show()

---
## 5. Summary

| Section | Best Approach | Key Result |
|---------|--------------|------------|
| Anomaly Detection | Isolation Forest | 167 anomalies (5%) detected |
| Risk Scorecard | Weighted rules | Very High Risk tier shows highest churn rate |
| Fraud Detection | Gradient Boosting | ROC-AUC=0.94, Avg Precision=0.91 |
| Threshold | Optimised (0.27) | 471 flagged, 100% precision, $30,845 at risk |

> **Action:** Deploy Gradient Boosting at threshold 0.27 in CRM. Run nightly batch scoring. Triage flagged customers by band for targeted intervention.
